In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def set_style(base=13, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base+2, "axes.labelsize": base, "xtick.labelsize": base-2,
        "ytick.labelsize": base-2, "legend.fontsize": base-2, "svg.fonttype": "none"})
set_style(base=16)

FIG4 = "../../../data/analysis/Fig4"
AF   = f"{FIG4}/all_features"
GTV5 = "../../../data/rBMRB/cluster_30p/PANAV/test.json"
FIG  = "Figs"

# ---- RECOMPUTE abserr from the v5 seed-ensemble predictions (same recipe as Grouping_geom.ipynb) ----
# "mean"/"mask" are the 10-seed ensemble mean and its own validity mask (tag/outlier exclusion is
# already folded into "mask" upstream -- do NOT re-apply a separate per-protein mean-centering or
# 4sigma-BMRB gate on top of it). pred/mask are right-padded to the batch's longest sequence, so
# truncate to len(gt_seq) instead of skipping the whole protein when longer; only skip if shorter.
pred = json.load(open(f"{FIG4}/results/predictions_N_seed_ensemble_10seed.json"))
gt   = json.load(open(GTV5))
mean_d, mask_d = pred["mean"], pred["mask"]
key_of = {v: k for k, v in gt["entry_ID"].items()}

ae_rows = []
length_mismatch = 0
for e, key in key_of.items():
    if e not in mean_d:
        continue
    seq, Nv = gt["sequence"][key], gt["N"][key]
    n = len(seq)
    pred_arr_full = np.asarray(mean_d[e], dtype=object)
    mask_full = np.asarray(mask_d[e], dtype=bool)
    if len(pred_arr_full) < n or len(mask_full) < n:
        length_mismatch += 1
        continue
    pm, msk = pred_arr_full[:n], mask_full[:n]

    base = np.array([r != "P" for r in seq]) & np.array([g is not None for g in Nv]) & msk
    if base.sum() == 0:
        continue
    t = np.array(Nv, dtype=object)[base].astype(float); pf = pm[base].astype(float)
    fidx = np.where(base)[0]
    err = abs(pf - t)
    for fi, a in zip(fidx, err):
        ae_rows.append((e, int(fi) + 1, float(a)))
ae = pd.DataFrame(ae_rows, columns=["entry", "resnum", "abserr"])
print(f"abserr | rows={len(ae)} | proteins={ae.entry.nunique()} (of {len(key_of)} entries in the v5 test set, "
      f"{length_mismatch} skipped for pred shorter than gt)")

# ---- per-residue pack_N8 + exposure from all_features/per_residue_records_full.json ----
# (same unified source as Grouping_geom.ipynb -- pack_N8 and exposure are already computed there
# alongside everything else, so the old separate geom_features_af2.csv / plddt_per_residue.json
# merges are no longer needed)
prr = json.load(open(f"{AF}/per_residue_records_full.json"))
rows = [(e, r["resnum"], r["pack_N8"], r["exposure"])
        for e, rl in prr.items() for r in rl
        if pd.notna(r.get("pack_N8")) and r.get("exposure") is not None]
feat = pd.DataFrame(rows, columns=["entry", "resnum", "pack_N8", "exposure"])

df = ae.merge(feat, on=["entry", "resnum"], how="inner")
print(f"pack_N8 x abserr | rows={len(df)} | proteins={df.entry.nunique()} (of {ae.entry.nunique()} proteins with abserr)")

In [ ]:
# ---- correlation between local packing (pack_N8) and |error| ----
# Only pack_N8 & abserr are used; dropna is restricted to these two columns, so missing
# values in any OTHER feature never remove a row here.
from scipy.stats import spearmanr, pearsonr

d = df.dropna(subset=["pack_N8", "abserr"])[["entry", "pack_N8", "abserr"]].copy()
print(f"kept {len(d)} / {len(df)} rows (dropped {len(df)-len(d)} lacking pack_N8) | proteins={d.entry.nunique()}")

x = d["pack_N8"].to_numpy(float); y = d["abserr"].to_numpy(float)
rho_s = spearmanr(x, y).correlation; rho_p = pearsonr(x, y)[0]
print(f"Spearman rho = {rho_s:+.3f}   Pearson r = {rho_p:+.3f}   (naive, ignores protein clustering)")

# protein-cluster bootstrap 95% CI for Spearman
# ent = d["entry"].to_numpy(); uniq = np.unique(ent); pidx = {e: np.where(ent == e)[0] for e in uniq}
# rng = np.random.default_rng(0); boot = []
# for _ in range(2000):
#     idx = np.concatenate([pidx[e] for e in rng.choice(uniq, uniq.size, replace=True)])
#     boot.append(spearmanr(x[idx], y[idx]).correlation)
# boot = np.array(boot); lo, hi = np.percentile(boot, [2.5, 97.5])
# p_two = 2 * min((boot <= 0).mean(), (boot >= 0).mean())
# print(f"Spearman rho = {rho_s:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (protein-cluster bootstrap)  p = {p_two:.3g}")

In [ ]:
# ---- binned curve: |error| vs local packing pack_N8 ----
import matplotlib.pyplot as plt

NBINS = 10
d["bin"] = pd.qcut(d.pack_N8, NBINS, duplicates="drop")     # equal-count bins -> stable medians
g = d.groupby("bin", observed=True)
xc  = g.pack_N8.median().values
med = g.abserr.median().values
q25 = g.abserr.quantile(0.25).values
q75 = g.abserr.quantile(0.75).values
ns  = g.abserr.size().values

# # # 95% bootstrap CI of each bin's median (resample residues, 2000x) -- illustrative band
# rng2 = np.random.default_rng(0); lo_ci, hi_ci = [], []
# for _, sub in g:
#     v = sub.abserr.to_numpy()
#     bs = [np.median(rng2.choice(v, v.size, replace=True)) for _ in range(2000)]
#     lo_ci.append(np.percentile(bs, 2.5)); hi_ci.append(np.percentile(bs, 97.5))
# lo_ci, hi_ci = np.array(lo_ci), np.array(hi_ci)

# print("median |error| by pack_N8 bin:")
# print(pd.DataFrame({"pack_med": xc.round(1), "err_med": med.round(3), "n": ns}).to_string(index=False))

In [ ]:
# ---- caption numbers for the binned pack_N8 curve (uses d / g / xc / med / q25 / q75 / ns from the cell above) ----
cap = pd.DataFrame({
    "bin": [str(b) for b in g.groups.keys()],
    "pack_min": g.pack_N8.min().values, "pack_max": g.pack_N8.max().values,
    "pack_med": xc, "err_q25": q25, "err_med": med, "err_q75": q75, "n": ns,
})
print(cap.round({"pack_med": 1, "err_q25": 3, "err_med": 3, "err_q75": 3}).to_string(index=False))

print(f"\nn residues = {len(d)}  |  proteins = {d.entry.nunique()}  |  bins = {len(xc)}")
print(f"pack_N8 range of bin medians: {xc.min():.0f} - {xc.max():.0f}  "
      f"(raw pack_N8 range {d.pack_N8.min():.0f} - {d.pack_N8.max():.0f})")
print(f"lowest bin : pack_N8 ~ {xc[0]:.0f}  median |error| = {med[0]:.2f} ppm")
print(f"highest bin: pack_N8 ~ {xc[-1]:.0f}  median |error| = {med[-1]:.2f} ppm")
print(f"bins 2-{len(xc)-1} (pack_N8 ~ {xc[1]:.0f}-{xc[-2]:.0f}): median |error| "
      f"{med[1:-1].min():.2f}-{med[1:-1].max():.2f} ppm")
print(f"bins with n per bin: {ns.min()}-{ns.max()}")


In [ ]:
set_style(16)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.fill_between(xc, q25, q75, color="#61AFEF", alpha=0.18, label="IQR (25-75%)")
# ax.fill_between(xc, lo_ci, hi_ci, color="#61AFEF", alpha=0.35, label="95% CI of median")
ax.plot(xc, med, "-o", color="#1f6fb2", lw=2, ms=6, label="median |error|")
ax.set_xlabel("pack_N8 (heavy atoms within 8 Å of N)")
ax.set_ylabel("|error| (ppm)")
ax.set_title("Per-residue |error| by pack_N8 bin (n = {})".format(len(d)))
ax.legend(frameon=False, loc="upper left")
# [ax.spines[s].set_visible(False) for s in ("top", "right")]
# ax.grid(alpha=0.3, ls="--")
plt.tight_layout()
fig.savefig(f"{FIG}/pack_N8_vs_error_binned.svg", bbox_inches="tight")
fig.savefig(f"{FIG}/pack_N8_vs_error_binned.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved pack_N8_vs_error_binned.svg/.png")

In [ ]:
# # ---- pack_N8 vs error, analysed by solvent exposure (buried vs exposed) ----
# from scipy.stats import spearmanr

# de = df.dropna(subset=["pack_N8","abserr","exposure"])[["entry","pack_N8","abserr","exposure"]].copy()
# de = de[de.exposure.isin(["buried","exposed"])]   # drops "proline" (no amide N-H to expose/bury)

# print("median pack_N8 / |err| by exposure:")
# print(de.groupby("exposure").agg(pack_med=("pack_N8","median"), err_med=("abserr","median"),
#                                  n=("abserr","size")).round(2).to_string())

# # packing is NOT equivalent to burial: buried fraction climbs gradually across pack_N8
# de["pbin"] = pd.qcut(de.pack_N8, 10, duplicates="drop")
# gb = de.groupby("pbin", observed=True)
# frac = pd.DataFrame({"pack_med": gb.pack_N8.median().round(0),
#                      "frac_buried": gb.exposure.apply(lambda s:(s=="buried").mean()).round(2),
#                      "err_med": gb.abserr.median().round(3), "n": gb.size()})
# print("\nper pack_N8 bin: buried fraction + median error (packing != burial)")
# print(frac.to_string(index=False))

# # within-exposure Spearman: does packing still predict error inside each class?
# def cluster_rho(sub, B=2000):
#     x, y, ent = sub.pack_N8.values, sub.abserr.values, sub.entry.values
#     rho = spearmanr(x, y).correlation
#     uniq = np.unique(ent); pidx = {e: np.where(ent==e)[0] for e in uniq}
#     rng = np.random.default_rng(0); b = []
#     for _ in range(B):
#         idx = np.concatenate([pidx[e] for e in rng.choice(uniq, uniq.size, replace=True)])
#         b.append(spearmanr(x[idx], y[idx]).correlation)
#     return rho, np.percentile(b,2.5), np.percentile(b,97.5)
# print("\nwithin-exposure Spearman(pack_N8, |error|), protein-cluster 95% CI:")
# for ex in ["exposed","buried"]:
#     r, lo, hi = cluster_rho(de[de.exposure==ex])
#     print(f"  {ex:8s} rho={r:+.3f}  [{lo:+.3f}, {hi:+.3f}]  (n={(de.exposure==ex).sum()})")

# # ---- plot: median |err| vs pack_N8 per exposure, median line + 95% CI-of-median band ----
# STYLE = {"exposed": ("#1f6fb2", "#61AFEF"), "buried": ("#b5303c", "#E06C75")}   # (line, fill)
# NB = 10
# rng = np.random.default_rng(0)
# set_style(14)
# fig, ax = plt.subplots(figsize=(7.6, 5.2))
# for ex in ["exposed", "buried"]:
#     s = de[de.exposure==ex].copy(); s["b"] = pd.qcut(s.pack_N8, NB, duplicates="drop")
#     g = s.groupby("b", observed=True)
#     xc = g.pack_N8.median().values; med = g.abserr.median().values
#     lo, hi = [], []
#     for _, sub in g:
#         v = sub.abserr.to_numpy()
#         bs = [np.median(rng.choice(v, v.size, replace=True)) for _ in range(2000)]
#         lo.append(np.percentile(bs, 2.5)); hi.append(np.percentile(bs, 97.5))
#     line, fill = STYLE[ex]
#     ax.fill_between(xc, lo, hi, color=fill, alpha=0.30, lw=0)
#     ax.plot(xc, med, "-o", color=line, lw=2.5, ms=8, mfc=line, mec="white", mew=1.2,
#             label=f"{ex}  (n={len(s)})", zorder=3)
# ax.set_xlabel("local packing  pack_N8  (heavy atoms within 8 Å of N)")
# ax.set_ylabel("|error| (ppm)")
# ax.legend(frameon=False, loc="lower right", title="shaded = 95% CI of median")
# # ax.grid(alpha=0.35, ls="--")
# [ax.spines[s].set_visible(False) for s in ("top", "right")]
# plt.tight_layout()
# fig.savefig(f"{FIG}/pack_N8_vs_error_by_exposure.svg", bbox_inches="tight")
# fig.savefig(f"{FIG}/pack_N8_vs_error_by_exposure.png", dpi=300, bbox_inches="tight")
# plt.show()
# print("saved pack_N8_vs_error_by_exposure.svg/.png")